# Multi-Hop Lineage Explorer
Recursively traverse upstream and downstream data lineage for any Unity Catalog table.

This notebook provides **two independent approaches** — run either section on its own:

| | Section 1: System Table | Section 2: REST API (Hybrid) |
| --- | --- | --- |
| **Source** | `system.access.table_lineage` | `/api/2.0/lineage-tracking/table-lineage` + system table seeds |
| **Freshness** | Lags by several hours | Near-real-time (same as UI) |
| **Dropped objects** | Filtered via `information_schema` | Filtered via `information_schema` |
| **Multi-hop** | SQL recursive CTE | Python BFS traversal |
| **Performance** | Single SQL query; scales well for large graphs | N+1 API calls (one per node); slower for wide/deep graphs |
| **Cost** | One Spark SQL job per run | Minimal compute (lightweight REST calls) + small Spark queries for seeds |
| **Best for** | Batch analytics, SQL joins, historical analysis | Real-time checks, matching UI lineage graph |

**Shared Parameters** (cell below):
- `TARGET_TABLE`: Fully qualified table name (e.g., `catalog.schema.table`)
- `MAX_DEPTH`: Maximum number of hops to traverse (default: 10)
- `LOOKBACK_DAYS`: Only consider lineage events from the last N days (default: 365, Section 1 only)
- `DIRECTION`: `both`, `upstream`, or `downstream`

In [0]:
# ============================================================
# CONFIGURATION - Modify these parameters for your use case
# ============================================================

# The fully qualified table name to analyze lineage for
TARGET_TABLE = "dkushari_uc.kroger_schema_1.nyctrip"

# Maximum number of hops to traverse (controls compute cost)
MAX_DEPTH = 10

# Only consider lineage events from the last N days
LOOKBACK_DAYS = 365

# Direction: 'both', 'upstream', or 'downstream'
DIRECTION = "both"

print(f"Target Table: {TARGET_TABLE}")
print(f"Max Depth:    {MAX_DEPTH}")
print(f"Lookback:     {LOOKBACK_DAYS} days")
print(f"Direction:    {DIRECTION}")

Target Table: dkushari_uc.kroger_schema_1.nyctrip
Max Depth:    10
Lookback:     365 days
Direction:    both


---
## Section 1: System Table Approach
Uses `system.access.table_lineage` with recursive CTEs for multi-hop traversal.

> **⚠ Latency Note:** This system table is populated asynchronously and can lag behind the Catalog Explorer UI by several hours. Newly created lineage relationships may not appear until the pipeline catches up.

In [0]:
def get_multi_hop_lineage(target_table: str, max_depth: int = 10, lookback_days: int = 90, direction: str = "both"):
    """
    Retrieve multi-hop upstream and/or downstream lineage for a given table.
    
    Uses SQL recursive CTEs with cycle detection (array_contains) to traverse
    the lineage graph from system.access.table_lineage.
    Filters out dropped/non-existent objects by joining against
    system.information_schema.tables.
    
    Args:
        target_table: Fully qualified table name (catalog.schema.table)
        max_depth: Maximum number of hops to traverse (default: 10)
        lookback_days: Only consider lineage events from last N days (default: 90)
        direction: 'both', 'upstream', or 'downstream'
    
    Returns:
        Spark DataFrame with columns: table_name, table_type, direction, 
        min_hop_level, max_hop_level, sample_path
    """
    
    downstream_cte = """
    downstream AS (
        SELECT 
            e.target AS table_name,
            e.target_type AS table_type,
            '{target}' AS root_table,
            1 AS hop_level,
            'downstream' AS direction,
            ARRAY('{target}', e.target) AS path
        FROM edges e
        WHERE e.source = '{target}'

        UNION ALL

        SELECT 
            e.target AS table_name,
            e.target_type AS table_type,
            d.root_table,
            d.hop_level + 1 AS hop_level,
            'downstream' AS direction,
            array_append(d.path, e.target) AS path
        FROM edges e
        INNER JOIN downstream d ON e.source = d.table_name
        WHERE d.hop_level < {max_depth}
            AND NOT array_contains(d.path, e.target)
    )""".format(target=target_table, max_depth=max_depth)
    
    upstream_cte = """
    upstream AS (
        SELECT 
            e.source AS table_name,
            e.source_type AS table_type,
            '{target}' AS root_table,
            1 AS hop_level,
            'upstream' AS direction,
            ARRAY('{target}', e.source) AS path
        FROM edges e
        WHERE e.target = '{target}'

        UNION ALL

        SELECT 
            e.source AS table_name,
            e.source_type AS table_type,
            u.root_table,
            u.hop_level + 1 AS hop_level,
            'upstream' AS direction,
            array_append(u.path, e.source) AS path
        FROM edges e
        INNER JOIN upstream u ON e.target = u.table_name
        WHERE u.hop_level < {max_depth}
            AND NOT array_contains(u.path, e.source)
    )""".format(target=target_table, max_depth=max_depth)

    # Build the query based on direction
    edges_cte = """
    edges AS (
        SELECT DISTINCT
            source_table_full_name AS source,
            target_table_full_name AS target,
            source_type,
            target_type
        FROM system.access.table_lineage
        WHERE source_table_full_name IS NOT NULL 
            AND target_table_full_name IS NOT NULL
            AND source_table_full_name <> target_table_full_name
            AND event_time >= current_timestamp() - INTERVAL {lookback} DAYS
    )""".format(lookback=lookback_days)

    # CTE to get all existing tables/views for filtering out dropped objects
    existing_tables_cte = """
    existing_tables AS (
        SELECT 
            concat(table_catalog, '.', table_schema, '.', table_name) AS full_name
        FROM system.information_schema.tables
    )"""

    if direction == "downstream":
        ctes = f"{edges_cte},\n{downstream_cte},\n{existing_tables_cte}"
        final_select = "SELECT * FROM downstream"
    elif direction == "upstream":
        ctes = f"{edges_cte},\n{upstream_cte},\n{existing_tables_cte}"
        final_select = "SELECT * FROM upstream"
    else:  # both
        ctes = f"{edges_cte},\n{downstream_cte},\n{upstream_cte},\n{existing_tables_cte}"
        final_select = "SELECT * FROM downstream UNION ALL SELECT * FROM upstream"

    query = f"""
    WITH RECURSIVE
    {ctes}
    
    SELECT 
        r.table_name,
        r.table_type,
        r.direction,
        min(r.hop_level) AS min_hop_level,
        max(r.hop_level) AS max_hop_level,
        first(r.path) AS sample_path
    FROM ({final_select}) r
    INNER JOIN existing_tables et ON r.table_name = et.full_name
    GROUP BY r.table_name, r.table_type, r.direction
    ORDER BY r.direction, min_hop_level, r.table_name
    """
    
    return spark.sql(query)

In [0]:
# Run the multi-hop lineage traversal
lineage_df = get_multi_hop_lineage(TARGET_TABLE, MAX_DEPTH, LOOKBACK_DAYS, DIRECTION)

total = lineage_df.count()
print(f"\nFound {total} related tables for '{TARGET_TABLE}'")
display(lineage_df)


Found 7 related tables for 'dkushari_uc.kroger_schema_1.nyctrip'


table_name,table_type,direction,min_hop_level,max_hop_level,sample_path
dkushari_uc.kroger_schema_1.nyctrip_tbl_frm_tbl_1,TABLE,downstream,1,1,"List(dkushari_uc.kroger_schema_1.nyctrip, dkushari_uc.kroger_schema_1.nyctrip_tbl_frm_tbl_1)"
dkushari_uc.kroger_schema_2.nyctrip_vw,VIEW,downstream,1,1,"List(dkushari_uc.kroger_schema_1.nyctrip, dkushari_uc.kroger_schema_2.nyctrip_vw)"
dkushari_uc.kroger_schema_1.nyctrip_tbl_frm_vw_1,TABLE,downstream,2,2,"List(dkushari_uc.kroger_schema_1.nyctrip, dkushari_uc.kroger_schema_2.nyctrip_vw, dkushari_uc.kroger_schema_1.nyctrip_tbl_frm_vw_1)"
dkushari_uc.kroger_schema_1.nyctrip_vw_frm_vw_2,VIEW,downstream,2,2,"List(dkushari_uc.kroger_schema_1.nyctrip, dkushari_uc.kroger_schema_2.nyctrip_vw, dkushari_uc.kroger_schema_1.nyctrip_vw_frm_vw_2)"
dkushari_uc.kroger_schema_2.nyctrip_tbl_frm_vw,TABLE,downstream,2,2,"List(dkushari_uc.kroger_schema_1.nyctrip, dkushari_uc.kroger_schema_2.nyctrip_vw, dkushari_uc.kroger_schema_2.nyctrip_tbl_frm_vw)"
dkushari_uc.kroger_schema_2.nyctrip_tbl_frm_vw_1,TABLE,downstream,3,3,"List(dkushari_uc.kroger_schema_1.nyctrip, dkushari_uc.kroger_schema_2.nyctrip_vw, dkushari_uc.kroger_schema_1.nyctrip_tbl_frm_vw_1, dkushari_uc.kroger_schema_2.nyctrip_tbl_frm_vw_1)"
dkushari_uc.demodb.nyctrip,TABLE,upstream,1,1,"List(dkushari_uc.kroger_schema_1.nyctrip, dkushari_uc.demodb.nyctrip)"


In [0]:
from pyspark.sql import functions as F

if lineage_df.count() > 0:
    summary = lineage_df.groupBy("direction").agg(
        F.count("*").alias("total_tables"),
        F.max("max_hop_level").alias("max_depth_reached"),
        F.countDistinct("table_type").alias("distinct_asset_types"),
        F.collect_set("table_type").alias("asset_types")
    )
    display(summary)

    # Breakdown by hop level
    hop_breakdown = lineage_df.groupBy("direction", "min_hop_level").agg(
        F.count("*").alias("tables_at_hop"),
        F.collect_list("table_name").alias("tables")
    ).orderBy("direction", "min_hop_level")
    
    print("\nBreakdown by hop level:")
    display(hop_breakdown)
else:
    print("No lineage data found. Try increasing LOOKBACK_DAYS or check if the table name is correct.")

direction,total_tables,max_depth_reached,distinct_asset_types,asset_types
upstream,1,1,1,List(TABLE)
downstream,6,3,2,"List(VIEW, TABLE)"



Breakdown by hop level:


direction,min_hop_level,tables_at_hop,tables
downstream,1,2,"List(dkushari_uc.kroger_schema_2.nyctrip_vw, dkushari_uc.kroger_schema_1.nyctrip_tbl_frm_tbl_1)"
downstream,2,3,"List(dkushari_uc.kroger_schema_1.nyctrip_vw_frm_vw_2, dkushari_uc.kroger_schema_1.nyctrip_tbl_frm_vw_1, dkushari_uc.kroger_schema_2.nyctrip_tbl_frm_vw)"
downstream,3,1,List(dkushari_uc.kroger_schema_2.nyctrip_tbl_frm_vw_1)
upstream,1,1,List(dkushari_uc.demodb.nyctrip)


---
## Section 2: REST API Approach (Hybrid)
Uses the Unity Catalog lineage tracking REST API (`/api/2.0/lineage-tracking/table-lineage`) with BFS traversal for **real-time multi-hop expansion**.

Direct (hop-1) relationships are seeded from both the REST API and `system.access.table_lineage` to catch older lineage events (e.g., historical CTAS operations) that the API alone may not return. Dropped objects are filtered via `information_schema`.

In [0]:
import requests
from collections import deque

def get_lineage_realtime(target_table: str, max_depth: int = 10, direction: str = "both"):
    """
    Retrieve multi-hop lineage using a hybrid approach:
    1. Seeds from BOTH the REST API and system.access.table_lineage
       for direct (hop-1) relationships — catches older lineage the API misses.
    2. Uses BFS over the REST API for multi-hop expansion (real-time).
    
    Args:
        target_table: Fully qualified table name (catalog.schema.table)
        max_depth: Maximum number of hops to traverse (default: 10)
        direction: 'both', 'upstream', or 'downstream'
    
    Returns:
        Spark DataFrame with columns: table_name, table_type, direction,
        hop_level, lineage_timestamp
    """
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host = ctx.apiUrl().get()
    token = ctx.apiToken().get()
    headers = {"Authorization": f"Bearer {token}"}
    
    def _fetch_one_hop(table_name):
        """Call the lineage API for a single table, return (upstreams, downstreams)."""
        resp = requests.get(
            f"{host}/api/2.0/lineage-tracking/table-lineage",
            headers=headers,
            params={"table_name": table_name}
        )
        if resp.status_code != 200:
            return [], []
        data = resp.json()
        upstreams = [
            {
                "table_name": f"{t['tableInfo']['catalog_name']}.{t['tableInfo']['schema_name']}.{t['tableInfo']['name']}",
                "table_type": t["tableInfo"].get("table_type", "UNKNOWN"),
                "lineage_timestamp": t["tableInfo"].get("lineage_timestamp"),
            }
            for t in data.get("upstreams", [])
            if "tableInfo" in t
        ]
        downstreams = [
            {
                "table_name": f"{t['tableInfo']['catalog_name']}.{t['tableInfo']['schema_name']}.{t['tableInfo']['name']}",
                "table_type": t["tableInfo"].get("table_type", "UNKNOWN"),
                "lineage_timestamp": t["tableInfo"].get("lineage_timestamp"),
            }
            for t in data.get("downstreams", [])
            if "tableInfo" in t
        ]
        return upstreams, downstreams

    def _get_system_table_seeds(table_name, dir_filter):
        """Query system.access.table_lineage for direct relationships the API may miss."""
        seeds = []
        try:
            if dir_filter in ("both", "upstream"):
                rows = spark.sql(f"""
                    SELECT source_table_full_name AS name, source_type AS type,
                           CAST(MAX(event_time) AS STRING) AS latest_event
                    FROM system.access.table_lineage
                    WHERE target_table_full_name = '{table_name}'
                      AND source_table_full_name IS NOT NULL
                      AND source_table_full_name <> '{table_name}'
                    GROUP BY source_table_full_name, source_type
                """).collect()
                for r in rows:
                    seeds.append({"table_name": r["name"], "table_type": r["type"] or "UNKNOWN",
                                  "lineage_timestamp": r["latest_event"], "direction": "upstream"})
            if dir_filter in ("both", "downstream"):
                rows = spark.sql(f"""
                    SELECT target_table_full_name AS name, target_type AS type,
                           CAST(MAX(event_time) AS STRING) AS latest_event
                    FROM system.access.table_lineage
                    WHERE source_table_full_name = '{table_name}'
                      AND target_table_full_name IS NOT NULL
                      AND target_table_full_name <> '{table_name}'
                    GROUP BY target_table_full_name, target_type
                """).collect()
                for r in rows:
                    seeds.append({"table_name": r["name"], "table_type": r["type"] or "UNKNOWN",
                                  "lineage_timestamp": r["latest_event"], "direction": "downstream"})
        except Exception:
            pass  # Fall back to API-only if system table is inaccessible
        return seeds

    # ------------------------------------------------------------------
    # Step 1: Seed from REST API (hop 1)
    # ------------------------------------------------------------------
    results = []
    visited = {target_table}
    queue = deque()  # BFS queue: (table_name, current_depth, direction)

    api_upstreams, api_downstreams = _fetch_one_hop(target_table)

    if direction in ("both", "downstream"):
        for n in api_downstreams:
            if n["table_name"] not in visited:
                visited.add(n["table_name"])
                results.append({**n, "direction": "downstream", "hop_level": 1})
                queue.append((n["table_name"], 1, "downstream"))
    if direction in ("both", "upstream"):
        for n in api_upstreams:
            if n["table_name"] not in visited:
                visited.add(n["table_name"])
                results.append({**n, "direction": "upstream", "hop_level": 1})
                queue.append((n["table_name"], 1, "upstream"))

    # ------------------------------------------------------------------
    # Step 2: Supplement hop 1 with system table seeds (catches older lineage)
    # ------------------------------------------------------------------
    sys_seeds = _get_system_table_seeds(target_table, direction)
    # Filter to existing tables only
    existing = set()
    try:
        seed_names = [s["table_name"] for s in sys_seeds if s["table_name"] not in visited]
        if seed_names:
            names_sql = ", ".join([f"'{n}'" for n in seed_names])
            existing = {r["full_name"] for r in spark.sql(f"""
                SELECT concat(table_catalog, '.', table_schema, '.', table_name) AS full_name
                FROM system.information_schema.tables
                WHERE concat(table_catalog, '.', table_schema, '.', table_name) IN ({names_sql})
            """).collect()}
    except Exception:
        existing = set(seed_names)  # Skip filter if information_schema fails

    for seed in sys_seeds:
        name = seed["table_name"]
        if name not in visited and name in existing:
            visited.add(name)
            results.append({
                "table_name": name, "table_type": seed["table_type"],
                "direction": seed["direction"], "hop_level": 1,
                "lineage_timestamp": seed["lineage_timestamp"],
            })
            queue.append((name, 1, seed["direction"]))

    # ------------------------------------------------------------------
    # Step 3: BFS multi-hop expansion via REST API (hop 2+)
    # ------------------------------------------------------------------
    while queue:
        current_table, depth, dir_to_explore = queue.popleft()
        if depth >= max_depth:
            continue

        upstreams, downstreams = _fetch_one_hop(current_table)
        neighbors = downstreams if dir_to_explore == "downstream" else upstreams

        for neighbor in neighbors:
            name = neighbor["table_name"]
            if name not in visited:
                visited.add(name)
                results.append({
                    "table_name": name, "table_type": neighbor["table_type"],
                    "direction": dir_to_explore, "hop_level": depth + 1,
                    "lineage_timestamp": neighbor["lineage_timestamp"],
                })
                queue.append((name, depth + 1, dir_to_explore))

    if not results:
        schema = "table_name STRING, table_type STRING, direction STRING, hop_level INT, lineage_timestamp STRING"
        return spark.createDataFrame([], schema)

    return spark.createDataFrame(results)

print("Hybrid lineage function ready (REST API + system table seeds).")

Hybrid lineage function ready (REST API + system table seeds).


In [0]:
# Run the real-time lineage traversal via REST API
rt_lineage_df = get_lineage_realtime(TARGET_TABLE, MAX_DEPTH, DIRECTION)

total = rt_lineage_df.count()
print(f"\nFound {total} related tables for '{TARGET_TABLE}' (real-time)")
display(rt_lineage_df.orderBy("direction", "hop_level", "table_name"))


Found 7 related tables for 'dkushari_uc.kroger_schema_1.nyctrip' (real-time)


direction,hop_level,lineage_timestamp,table_name,table_type
downstream,1,2026-03-20 16:56:31.0,dkushari_uc.kroger_schema_1.nyctrip_tbl_frm_tbl_1,TABLE
downstream,1,2026-03-20 16:57:53.0,dkushari_uc.kroger_schema_2.nyctrip_vw,PERSISTED_VIEW
downstream,2,2026-03-20 16:54:45.0,dkushari_uc.kroger_schema_1.nyctrip_tbl_frm_vw_1,TABLE
downstream,2,2026-03-20 16:57:53.0,dkushari_uc.kroger_schema_1.nyctrip_vw_frm_vw_2,PERSISTED_VIEW
downstream,2,2026-03-20 16:54:20.0,dkushari_uc.kroger_schema_2.nyctrip_tbl_frm_vw,TABLE
downstream,3,2026-03-20 16:59:48.0,dkushari_uc.kroger_schema_2.nyctrip_tbl_frm_vw_1,TABLE
upstream,1,2025-04-07 17:12:24.733,dkushari_uc.demodb.nyctrip,TABLE


In [0]:
from pyspark.sql import functions as F

if rt_lineage_df.count() > 0:
    summary = rt_lineage_df.groupBy("direction").agg(
        F.count("*").alias("total_tables"),
        F.max("hop_level").alias("max_depth_reached"),
        F.countDistinct("table_type").alias("distinct_asset_types"),
        F.collect_set("table_type").alias("asset_types")
    )
    display(summary)

    # Breakdown by hop level
    hop_breakdown = rt_lineage_df.groupBy("direction", "hop_level").agg(
        F.count("*").alias("tables_at_hop"),
        F.collect_list("table_name").alias("tables")
    ).orderBy("direction", "hop_level")
    
    print("\nBreakdown by hop level:")
    display(hop_breakdown)
else:
    print("No lineage data found. Check if the table name is correct.")

direction,total_tables,max_depth_reached,distinct_asset_types,asset_types
downstream,6,3,2,"List(PERSISTED_VIEW, TABLE)"
upstream,1,1,1,List(TABLE)



Breakdown by hop level:


direction,hop_level,tables_at_hop,tables
downstream,1,2,"List(dkushari_uc.kroger_schema_2.nyctrip_vw, dkushari_uc.kroger_schema_1.nyctrip_tbl_frm_tbl_1)"
downstream,2,3,"List(dkushari_uc.kroger_schema_1.nyctrip_vw_frm_vw_2, dkushari_uc.kroger_schema_1.nyctrip_tbl_frm_vw_1, dkushari_uc.kroger_schema_2.nyctrip_tbl_frm_vw)"
downstream,3,1,List(dkushari_uc.kroger_schema_2.nyctrip_tbl_frm_vw_1)
upstream,1,1,List(dkushari_uc.demodb.nyctrip)


In [0]:
# ============================================================
# BATCH MODE: Analyze lineage for multiple tables at once
# Works with either approach - uncomment the one you prefer
# ============================================================

# tables_to_analyze = [
#     "catalog.schema.table1",
#     "catalog.schema.table2",
#     "catalog.schema.table3",
# ]
# 
# from functools import reduce
# from pyspark.sql import DataFrame
# 
# all_lineage = []
# for tbl in tables_to_analyze:
#     print(f"Processing: {tbl}")
#
#     # Option A: System Table approach (has lag, supports SQL analytics)
#     # df = get_multi_hop_lineage(tbl, MAX_DEPTH, LOOKBACK_DAYS, DIRECTION)
#
#     # Option B: REST API approach (real-time, matches UI)
#     # df = get_lineage_realtime(tbl, MAX_DEPTH, DIRECTION)
#
#     df = df.withColumn("analyzed_table", F.lit(tbl))
#     all_lineage.append(df)
# 
# if all_lineage:
#     combined = reduce(DataFrame.unionByName, all_lineage)
#     display(combined)